# NB-04 · Qualitative Analysis

Case studies showing pipeline complementarity and recommendation quality.

**Note**: This notebook uses pre-computed recommendation outputs rather than live model inference.
Running live inference requires the full model stack (FAISS index + DIF-SASRec weights + BGE-M3 on CUDA).
See `EXPERIMENT_ROADMAP.md` for instructions to run live inference.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils import setup_style, save_fig, FIG_DIR, ROOT, PALETTE

setup_style()

## 1. Complementarity Case Study

The following illustrates the four complementarity outcomes from the canonical run
(100k users, N=99, seed=42), sourced from the evaluation log.

In [ ]:
# Complementarity counts (from evaluation/logs/20260426_233351.log)
# INFO  Complementarity  A∩B=70,557  A-only=19,909  B-only=6,893  neither=2,641
comp = {'A∩B': 70557, 'A-only': 19909, 'B-only': 6893, 'Neither': 2641}
total = sum(comp.values())

print('Pipeline Complementarity (canonical run, 100k users)')
print('─' * 55)
for outcome, count in comp.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {outcome:10s}: {count:6,}  ({pct:5.1f}%)  {bar}')
print(f'  {"Total":10s}: {total:6,}')

covered = (total - comp['Neither']) / total * 100
print(f'\n  Coverage (≥1 pipeline hit): {covered:.1f}%')
print(f'  Unique-to-B (B recovers A misses): {comp["B-only"]:,} users ({comp["B-only"]/total*100:.1f}%)')

## 2. User Profile Types (qualitative illustration)

Three archetypal user behaviours and how the pipelines respond.

In [ ]:
# Illustrative user archetypes (representative, not live inference)
archetypes = [
    {
        'name': 'Type A — Dense Reader (A∩B)',
        'description': 'High-activity user with 500+ interactions spanning consistent genres.',
        'train_clicks': 512,
        'pipeline_a_hit': True,
        'pipeline_b_hit': True,
        'explanation': 'Both Cleora graph (co-purchase) and DIF-SASRec (sequential intent) '
                       'capture the strong genre preference. Both hit → coverage = 70.6% of users.',
    },
    {
        'name': 'Type B — Graph-Prominent (A only)',
        'description': 'User with eclectic taste; purchases co-occur densely in graph but sequence is noisy.',
        'train_clicks': 245,
        'pipeline_a_hit': True,
        'pipeline_b_hit': False,
        'explanation': 'Cleora captures co-purchase communities (e.g., gift-buying bursts). '
                       'DIF-SASRec is misled by the noisy sequence. A-only → 19.9% of users.',
    },
    {
        'name': 'Type C — Sequential Intent (B only)',
        'description': 'User who recently shifted interest; sequence captures the shift, graph does not.',
        'train_clicks': 78,
        'pipeline_a_hit': False,
        'pipeline_b_hit': True,
        'explanation': 'DIF-SASRec detects the recent intent change in the last 20 interactions. '
                       'Cleora index reflects older global co-purchase patterns. B-only → 6.9% of users.',
    },
]

for a in archetypes:
    a_hit = '✓' if a['pipeline_a_hit'] else '✗'
    b_hit = '✓' if a['pipeline_b_hit'] else '✗'
    print(f"{'─'*60}")
    print(f"  {a['name']}")
    print(f"  Interactions: {a['train_clicks']}  |  Pipeline A: {a_hit}  Pipeline B: {b_hit}")
    print(f"  {a['explanation']}")
print('─' * 60)

## 3. Figure: User Archetype Summary

In [ ]:
# Visualise complementarity breakdown with archetype annotation
fig, ax = plt.subplots(figsize=(8, 5))

outcomes = ['A∩B\n(both hit)', 'A only', 'B only', 'Neither']
counts   = [70557, 19909, 6893, 2641]
pcts     = [c / total * 100 for c in counts]
colors   = ['#C678DD', '#98C379', '#61AFEF', '#9E9E9E']
type_labels = ['Type A\n(dense readers)', 'Type B\n(graph-prominent)', 'Type C\n(sequential intent)', 'Unrecoverable']

bars = ax.bar(outcomes, counts, color=colors, edgecolor='white', linewidth=1.2)

for bar, c, p, lbl in zip(bars, counts, pcts, type_labels):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{c:,}\n({p:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.5,
            lbl, ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_ylabel('Number of users')
ax.set_title('Pipeline complementarity with user archetypes\n(100k users, N=99, canonical run)')
ax.set_ylim(0, max(counts) * 1.25)

save_fig('fig_qualitative_archetypes', fig)
plt.show()

## 4. Error Analysis: Where All Methods Fail

In [ ]:
# Users where neither pipeline hit (2,641 / 100k = 2.64%)
# These are the hardest cases. Potential reasons:

error_analysis = [
    ('Cold-start users (< 5 interactions)',
     'DIF-SASRec degrades on short sequences; Cleora index requires co-purchase history.'),
    ('Highly niche / long-tail items',
     'Test item not in the 375k Cleora index AND sequence too sparse for intent detection.'),
    ('Cross-domain interest shifts',
     'User abruptly switches genre (e.g., fiction → technical), causing both pipelines to miss.'),
    ('Content veto over-filtering',
     'DIF-SASRec candidate blocked by τ=0.3 cosine threshold (overly conservative).'),
]

print(f'"Neither" users: {comp["Neither"]:,} / {total:,} ({comp["Neither"]/total*100:.2f}%)')
print('Suspected failure modes:')
for i, (cause, explanation) in enumerate(error_analysis, 1):
    print(f'  {i}. {cause}')
    print(f'     {explanation}')